In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go

In [2]:
# Plot predictions - baseline against associated variables
# Ensure completeness on 2023 baseline data, 2023 and 2019 predictions, associations

# Load data
### Load measurement data

In [102]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# Load AC predictions
df_res = bd.load_meas_from_excel(
    # "infer_AR_using_19_23_data_2entries_fev1_10122025",
    "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
    study_folder="CFR",
    str_cols_to_arrays=["Airway resistance (%)"],
).drop(columns="Healthy FEV1 (L)")
df_res23 = df_res[df_res["Date Recorded"] == datetime.date(2023, 1, 1)]
print(f"Shape: {df_res23.shape}")

# Merge predictions and baseline data
meascols = ["ID", "Date Recorded", "ecFEV1 % Predicted"]
df = df_res23.merge(df_meas[meascols], on=["ID", "Date Recorded"])
print(f"Shape dfmeas + df_res23: {df.shape}")
# CCL: Results are complete which is expected because computed based on df_meas

Shape: (1485, 3)
Shape dfmeas + df_res23: (1485, 4)


In [ ]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
df = bd.load_meas_from_excel(
    "AR_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=["Airway resistance (%)"],
)
print(f"Shape: {df.shape}")

Shape: (2046, 18)


In [2]:
df = bd.load_meas_from_excel(
    "infer_all_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
)

### Load IV data

In [ ]:
# Load IV data from 2019-23
# df_ass = pd.read_excel(dh.get_path_to_main() + "ExcelFiles/CFR/IV_data_19-23.xlsx")
df_ass = pd.read_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/anbitiotics_data_19-23.xlsx"
)
df_ass["IVs"] = df_ass["Home IVs"] + df_ass["Hosp IVs"]

df_avg_ivs_per_year = (
    df_ass.groupby("ID")
    .agg(
        {
            "Home IVs": "mean",
            "Hosp IVs": "mean",
            "IVs": "mean",
            "Oral": "mean",
            "Any antibiotics": "mean",
        }
    )
    .rename(
        columns={
            "Home IVs": "Avg Home IVs",
            "Hosp IVs": "Avg Hosp IVs",
            "IVs": "Avg IVs",
            "Oral": "Avg Oral",
            "Any antibiotics": "Avg Any antibiotics",
        }
    )
).reset_index()

In [11]:
df_ass = bd.load_meas_from_excel(
    "IV_data_2019",
    study_folder="CFR",
)

### Create aggregated df

In [5]:
# Merge associated variables into the df
df = df.merge(df_avg_ivs_per_year, on="ID", how="left")
print(f"Shape final df: {df.shape}")

# Associations are complete

Shape final df: (2037, 26)


In [14]:
df = df.merge(df_ass, on=["ID", "Date Recorded"])

In [6]:
AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df[AC.name] = df[AR.name].apply(lambda arr: arr[::-1])
df["mean AC"] = df[AC.name].apply(lambda ac: AC.get_mean(ac))
df.head(1)

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,P(FEV1|pred_FEV1),P(FEV1_obs|pred_FEV1),Airway resistance (%),Avg Home IVs,Avg Hosp IVs,Avg IVs,Avg Oral,Avg Any antibiotics,Airway conductance (%),mean AC
0,B155916,32,162,1.5,0.47,1.64,Female,2019-01-01,1.5,0.47,...,"[1.92074181e-247, 2.64360202e-224, 5.15354678e...",0.000005,"[8.65509846e-09, 8.15554203e-08, 6.34526903e-0...",0.2,0.4,0.6,0.8,1.4,"[1.58215462e-103, 4.6856718e-81, 2.19166211e-5...",50.223704


# Exploring associations

In [19]:
df.describe()

,Age,Height,FEV1,FEF2575,best FEV1,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,...,Home IVs,Home IV days,Oral,Non Hosp IVs,Non Hosp IV days,Chest episodes,Cough episodes,Pulm Abscess,IVs,IV days
count,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,...,2037.000000,2037.000000,2022.000000,2037.000000,2037.000000,19.000000,3.0,3.000000,2037.000000,2037.000000
mean,32.446735,168.074129,2.373108,1.665654,2.564520,2.373108,1.665654,63.547092,3.669590,64.277982,...,0.756014,10.120275,1.850643,0.224350,0.545410,1.263158,1.0,1.666667,1.697595,20.831124
std,11.520706,9.464436,0.999659,1.209953,1.005902,0.999659,1.209953,28.087835,0.693666,22.933702,...,1.375790,24.408942,1.892313,0.566966,2.501969,0.561951,0.0,1.154701,2.413098,35.199878
min,18.000000,138.000000,0.400000,0.130000,0.460000,0.400000,0.130000,8.108108,1.747108,13.313079,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.0,1.000000,0.000000,0.000000
25%,23.000000,161.000000,1.580000,0.670000,1.770000,1.580000,0.670000,40.573770,3.092335,46.710225,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.0,1.000000,0.000000,0.000000
50%,30.000000,168.000000,2.340000,1.360000,2.530000,2.340000,1.360000,58.255451,3.577940,66.020756,...,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.0,1.000000,1.000000,10.000000
75%,39.000000,175.000000,3.020000,2.400000,3.240000,3.020000,2.400000,81.132074,4.230593,81.742912,...,1.000000,13.000000,3.000000,0.000000,0.000000,1.000000,1.0,2.000000,3.000000,29.000000
max,80.000000,197.000000,5.530000,7.000000,5.880000,5.530000,7.000000,224.352334,5.534266,128.519504,...,10.000000,332.000000,20.000000,6.000000,45.000000,3.000000,1.0,3.000000,20.000000,335.000000


In [20]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'P(HFEV1|FEF2575, bFEV1, FEV1)', 'P(HFEV1|FEV1)',
       'Airway resistance (%)', 'Hosp IVs', 'Hosp IV days', 'Home IVs',
       'Home IV days', 'Oral', 'Non Hosp IVs', 'Non Hosp IV days',
       'Chest episodes', 'Cough episodes', 'Pulm Abscess', 'Smoking status',
       '2nd hand smoking exposure', 'IVs', 'IV days'],
      dtype='object')

In [22]:
import src.models.helpers as mh

HFEV1 = mh.VariableNode("Healthy FEV1 (L)", 1, 6, 0.05, prior=None)

df["mean HFEV1_pers"] = df["P(HFEV1|FEF2575, bFEV1, FEV1)"].apply(
    lambda x: HFEV1.get_mean(x)
)
df["mean HFEV1_ST"] = df["P(HFEV1|FEV1)"].apply(lambda x: HFEV1.get_mean(x))
# Prediction: Compute FEV1 in percentage of the mean personalised HFEV1 value
df["FEV1%PersPred"] = df["FEV1"] / df["mean HFEV1_pers"] * 100
# Baseline: Compute FEV1 in percentage of the mean softly truncated HFEV1 value
df["FEV1%STPred"] = df["FEV1"] / df["mean HFEV1_ST"] * 100

In [29]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/pppfev1_ppfev1_IV_19_assoc.xlsx",
    index=False,
)

### Load data

In [3]:
df = bd.load_meas_from_excel(
    "pppfev1_ppfev1_IV_19_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
)

### Ranked correlation

#### Corr of pred/baseline vs IVs

In [4]:
# challenge: find correlation in X-Y with Z where X and Y are already correlated with Z
# X-Y = 5 don't have the same meaning whether severer of mild CF
# Stratify by disease severity level (or more granular), then check if diff is correlated with IVs

from scipy.stats import spearmanr

# Does it corrrelated better?
metrics = ["FEV1%PersPred", "FEV1%PredST"]
iv_days = ["IV days", "Non Hosp IV days"]
ivs = ["IVs", "Non Hosp IVs"]

for ass_var in iv_days + ivs:
    corr1, pval1 = spearmanr(df["FEV1%PersPred"], df[ass_var])
    corr2, pval2 = spearmanr(df["FEV1%PredST"], df[ass_var])
    print(f"corr(pppFEV1,{ass_var}): {corr1:.3f} ({pval1:.3f})")
    print(f"corr(stppFEV1,{ass_var}): {corr2:.3f} ({pval2:.3f})")

# FEV1% metrics correlated better with number of treatment days that number of treatments

corr(pppFEV1,IV days): -0.487 (0.000)
corr(stppFEV1,IV days): -0.490 (0.000)
corr(pppFEV1,Non Hosp IV days): -0.069 (0.002)
corr(stppFEV1,Non Hosp IV days): -0.068 (0.002)
corr(pppFEV1,IVs): -0.468 (0.000)
corr(stppFEV1,IVs): -0.469 (0.000)
corr(pppFEV1,Non Hosp IVs): -0.066 (0.003)
corr(stppFEV1,Non Hosp IVs): -0.065 (0.004)


In [ ]:
prctile = 50
t = df["pppFEV1 - ppFEV1"].abs().quantile(prctile / 100)
dftmp = df[df["pppFEV1 - ppFEV1"].abs() > t]
iv_col = "IV days"
title = f"ppFEV1 vs pppFEV1 coloured by {iv_col}, {dftmp.shape[0]} entries, {prctile}% biggest diff"
fig = vh.plot_sidebyside_scatter(dftmp, "FEV1%STPred", "FEV1%PersPred", iv_col, title)

#### Stratify per ppFEV1, corr(ppPFEV1, IVs)

In [ ]:
# Scatter: stratify by 10% bins (0-10, 10-20, ..., 90-100, 100+)
fev_col = "FEV1%STPred"

bin_edges = list(range(0, 101, 10)) + [float("inf")]
bin_labels = [f"{i}-{i+10}" for i in range(0, 100, 10)] + ["100+"]

df["FEV1_bin_10pct"] = pd.cut(
    df[fev_col],
    bins=bin_edges,
    labels=bin_labels,
    right=False,
    include_lowest=True,
)

# # Optional: dictionary of boolean masks per bin
# bin_masks = {label: (df["FEV1_bin_10pct"] == label) for label in bin_labels}

In [14]:
def get_corr_for_df(df):
    # Does it corrrelated better?
    metrics = ["FEV1%PersPred", "FEV1%STPred"]
    iv_days = ["IV days", "Non Hosp IV days"]

    for ass_var in iv_days:
        corr1, pval1 = spearmanr(df["FEV1%PersPred"], df[ass_var])
        corr2, pval2 = spearmanr(df["FEV1%STPred"], df[ass_var])
        print(f"corr(pppFEV1,{ass_var}): {corr1:.3f} ({pval1:.3f})")
        print(f"corr(stppFEV1,{ass_var}): {corr2:.3f} ({pval2:.3f})")


df.groupby("FEV1_bin_10pct").apply(lambda group: get_corr_for_df(group))

corr(pppFEV1,IV days): nan (nan)
corr(stppFEV1,IV days): nan (nan)
corr(pppFEV1,Non Hosp IV days): nan (nan)
corr(stppFEV1,Non Hosp IV days): nan (nan)
corr(pppFEV1,IV days): -0.121 (0.566)
corr(stppFEV1,IV days): -0.115 (0.583)
corr(pppFEV1,Non Hosp IV days): -0.028 (0.895)
corr(stppFEV1,Non Hosp IV days): -0.042 (0.841)
corr(pppFEV1,IV days): -0.201 (0.017)
corr(stppFEV1,IV days): -0.199 (0.018)
corr(pppFEV1,Non Hosp IV days): 0.039 (0.650)
corr(stppFEV1,Non Hosp IV days): 0.038 (0.658)
corr(pppFEV1,IV days): -0.183 (0.015)
corr(stppFEV1,IV days): -0.185 (0.014)
corr(pppFEV1,Non Hosp IV days): 0.001 (0.992)
corr(stppFEV1,Non Hosp IV days): -0.002 (0.978)
corr(pppFEV1,IV days): -0.158 (0.019)
corr(stppFEV1,IV days): -0.196 (0.004)
corr(pppFEV1,Non Hosp IV days): -0.014 (0.842)
corr(stppFEV1,Non Hosp IV days): -0.046 (0.498)
corr(pppFEV1,IV days): 0.017 (0.784)
corr(stppFEV1,IV days): -0.041 (0.498)
corr(pppFEV1,Non Hosp IV days): -0.018 (0.772)
corr(stppFEV1,Non Hosp IV days): 0.060 (

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_90529/3082580473.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_90529/3082580473.py:13: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



""


#### Dumbel plots with IV days overlay

In [5]:
prctile = 50
# prctile = 0

title = f"Dumbell plot for 2019 CFR data with best FEV1, {prctile:.0f}th prctile"

diff_col = "pppFEV1 - ppFEV1ST"
t = df[diff_col].abs().quantile(prctile / 100)
df_to_plot = df[df[diff_col].abs() > t]

fig = make_subplots(
    1, 3, horizontal_spacing=0.2, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

# Create the three dataframe
mask_mild = df_to_plot["FEV1%PredST"] >= 70
mask_moderate = (df_to_plot["FEV1%PredST"] >= 40) & (df_to_plot["FEV1%PredST"] < 70)
mask_severe = 40 > df_to_plot["FEV1%PredST"]

vh.plot_scalar_dumbell(fig, df_to_plot[mask_severe], "FEV1%PredST", "FEV1%PersPred", col=1)
vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_moderate], "FEV1%PredST", "FEV1%PersPred", col=2
)
vh.plot_scalar_dumbell(fig, df_to_plot[mask_mild], "FEV1%PredST", "FEV1%PersPred", col=3)

fig.update_layout(
    height=1400 if prctile == 0 else 800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(
    range=[-1, 101],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
# fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
fig.show()

# Exploring where model confidently disagrees with baseline

In [9]:
# Compute P(baseline|prediction)
df["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df, AC)
df["P(ppFEV1|AC ratioed)"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# NOTE: AC is undefined above 100%, where ppFEV1 > 100%, value is clipped to 100%

# Plot histogram of P(ppFEV1|AC)
fig = px.histogram(df, x="P(ppFEV1|AC ratioed)", nbins=100)
title = f"Probability of baseline FEV1%pred (clipped) given the predicted airway conductance ratioed"
fig.update_layout(title=title, width=800, height=400)
# fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
fig.show()

In [10]:
# Is the ratioed version much different than the original?
idx_conf_disagree = df[df["P(ppFEV1|AC)"] <= t].index
idx_conf_disagree_ratioed = df[df["P(ppFEV1|AC) ratioed"] <= t_ratioed].index

NameError: name 't' is not defined

## Viz: antibiotics association where the model confidently disagrees with the baseline ppFEV1?

In [13]:
df["P(ppFEV1|AC) ratioed"] = df["P(ppFEV1|AC ratioed)"]

In [4]:
df["clipped ecFEV1%Predicted"] = df["ecFEV1 % Predicted"].clip(upper=100)

import numpy as np
import plotly.graph_objects as go

ivs_col = "Avg Hosp IVs"
ivs_col = "Avg Any antibiotics"

ratioed = True
prctile = 10
df_conf, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)

fig = go.Figure(
    data=go.Scatter(
        x=df_conf["clipped ecFEV1%Predicted"],
        y=df_conf["mean AC"],
        mode="markers",
        marker=dict(
            color=df_conf[ivs_col],
            colorbar=dict(title=ivs_col),
            colorscale="Turbo",
            # colorscale=[
            #     [0.0, "white"],
            #     [[i/(len(px.colors.sequential.Turbo)-1), c] for i, c in enumerate(px.colors.sequential.Turbo)],
            # ],
            # line=dict(
            #     color=np.where(df_conf[ivs_col] == 0, "black", "rgba(0,0,0,0)"),
            #     width=np.where(df_conf[ivs_col] == 0, 1.5, 0),
            # ),
        ),
        hovertemplate="ID: %{customdata[0]}<br>"
        + "clipped ecFEV1%Predicted: %{x}<br>"
        + f"{AC.name} (mean): "
        + "%{y}<br>"
        + f"{ivs_col}: "
        + "%{marker.color}<extra></extra>",
        customdata=np.stack([df_conf["ID"]], axis=-1),
    )
)
# Draw a proportional line (y = x) for reference
min_x = df_conf["clipped ecFEV1%Predicted"].min()
max_x = df_conf["clipped ecFEV1%Predicted"].max()
fig.add_trace(
    go.Scatter(
        x=[min_x, max_x],
        y=[min_x, max_x],
        mode="lines",
        line=dict(color="black", dash="dash"),
    )
)

if ratioed:
    title = f"Scatter plot of ppFEV1 vs mean AC coloured by {ivs_col}<br> P(ppFEV1|AC) {prctile:.0f}th prctile"
else:
    title = f"Scatter plot of ppFEV1 vs mean AC coloured by {ivs_col}<br> P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"

fig.update_layout(
    title=title,
    xaxis_title="FEV1%Predicted (clipped)",
    yaxis_title=AC.name,
    width=800,
    height=700,
    showlegend=False,
)

fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/Antibiotics associations/{title}.pdf")
fig.show()

KeyError: 'P(ppFEV1|AC) ratioed'

In [ ]:
[
    [i / (len(px.colors.sequential.Turbo) - 1), c]
    for i, c in enumerate(px.colors.sequential.Turbo)
]

[[0.0, '#30123b'],
 [0.07142857142857142, '#4145ab'],
 [0.14285714285714285, '#4675ed'],
 [0.21428571428571427, '#39a2fc'],
 [0.2857142857142857, '#1bcfd4'],
 [0.35714285714285715, '#24eca6'],
 [0.42857142857142855, '#61fc6c'],
 [0.5, '#a4fc3b'],
 [0.5714285714285714, '#d1e834'],
 [0.6428571428571429, '#f3c63a'],
 [0.7142857142857143, '#fe9b2d'],
 [0.7857142857142857, '#f36315'],
 [0.8571428571428571, '#d93806'],
 [0.9285714285714286, '#b11901'],
 [1.0, '#7a0402']]

In [15]:
# Superimposed scatter plot of ppFEV1 and AC vs antibiotics

fig = go.Figure()


dftmp = df_conf
# dftmp = df

# Scatter for model predicted mean AC
fig.add_trace(
    go.Scatter(
        x=dftmp["mean AC"],
        y=dftmp["Avg Any antibiotics"],
        mode="markers",
        name="Prediction",
        marker=dict(color="blue"),
        hovertemplate="mean AC: %{x}<br>Avg Any antibiotics: %{y}",
    )
)

# Scatter for baseline (clipped ecFEV1%Predicted)
fig.add_trace(
    go.Scatter(
        x=dftmp["clipped ecFEV1%Predicted"],
        y=dftmp["Avg Any antibiotics"],
        mode="markers",
        name="Baseline",
        marker=dict(color="red"),
        hovertemplate="clipped ecFEV1%Predicted: %{x}<br>Avg Any antibiotics: %{y}",
    )
)

fig.update_traces(marker=dict(size=4))

# Add arrows from baseline prediction (clipped ecFEV1%Predicted) to model output (mean AC)
for ix, row in dftmp.iterrows():
    fig.add_shape(
        type="line",
        x0=row["clipped ecFEV1%Predicted"],
        y0=row["Avg Any antibiotics"],
        x1=row["mean AC"],
        y1=row["Avg Any antibiotics"],
        line=dict(color="gray", width=1, dash="dot"),
        # opacity=1,
        # layer="below"
    )

if ratioed:
    title = f"Superimposed scatter plot of baseline and prediction vs # antibiotics <br> P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
else:
    title = f"Superimposed scatter plot of baseline and prediction vs # antibiotics <br> P(ppFEV1|AC) {prctile:.0f}th prctile"

fig.update_xaxes(title_text=AC.name)
fig.update_yaxes(title_text="Average number of antibiotics (IV or oral)")
fig.update_layout(
    height=800,
    width=800,
    title=title,
)
# fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/Antibiotics associations/{title}.pdf")

## Ranked correlations

In [15]:
df

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,Oral,Non Hosp IVs,Non Hosp IV days,Chest episodes,Cough episodes,Pulm Abscess,Smoking status,2nd hand smoking exposure,IVs,IV days
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,...,3.0,1,1,NaN,NaN,NaN,NK,N,3,28.0
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,...,2.0,1,1,NaN,NaN,NaN,N,NK,0,0.0
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,...,3.0,2,2,NaN,NaN,NaN,N,NK,2,13.0
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,...,3.0,0,0,NaN,NaN,NaN,N,NK,2,15.0
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,...,6.0,1,1,NaN,NaN,NaN,N,NK,5,61.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,C222738,59,158,1.66,1.18,1.95,Female,2019-01-01,1.66,1.18,...,0.0,0,0,NaN,NaN,NaN,N,NK,0,0.0
2033,C222739,69,153,1.63,0.66,1.63,Female,2019-01-01,1.63,0.66,...,6.0,0,0,NaN,NaN,NaN,N,NK,0,0.0
2034,C222741,33,162,1.71,0.86,1.77,Female,2019-01-01,1.71,0.86,...,4.0,0,0,NaN,NaN,NaN,N,NK,3,12.0
2035,C222780,27,176,3.54,4.15,3.54,Female,2019-01-01,3.54,4.15,...,0.0,0,0,NaN,NaN,NaN,N,N,0,0.0


In [ ]:
from scipy.stats import spearmanr

# Spearman correlation
dftmp = df
dftmp = df_conf

# dftmp["AC sampled"] = dftmp[AC.name].apply(lambda ac: AC.sample(n=50, p=ac))

for ab_col in ["Avg Any antibiotics", "Avg IVs", "Avg Oral"]:
    print(ab_col)
    corr_ecfev1, pval_ecfev1 = spearmanr(dftmp["ecFEV1 % Predicted"], dftmp[ab_col])
    print(f"Spearman corr with ppFEV1:  r={corr_ecfev1:.3f}, p={pval_ecfev1:.3g}")

    corr_acsampled, pval_acsampled = spearmanr(dftmp["mean AC"], dftmp[ab_col])
    # corr_acsampled, pval_acsampled = spearmanr(dftmp["AC sampled"].explode(), dftmp.loc[dftmp.index.repeat(50), ab_col].values)
    print(f"Spearman corr with mean AC: r={corr_acsampled:.3f}, p={pval_acsampled:.3g}")

Avg Any antibiotics
Spearman corr with ppFEV1:  r=-0.325, p=2.11e-06
Spearman corr with mean AC: r=-0.306, p=8.44e-06
Avg IVs
Spearman corr with ppFEV1:  r=-0.364, p=8.55e-08
Spearman corr with mean AC: r=-0.342, p=5.34e-07
Avg Oral
Spearman corr with ppFEV1:  r=-0.232, p=0.000849
Spearman corr with mean AC: r=-0.225, p=0.00122


## Dive into large diffs (for day1 2023 - day22019 data)

In [83]:
df_conf.columns

Index(['ID', 'best FEV1', 'Age', 'Height', 'FEV1', 'FEF2575', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted', 'idx FEV1',
       'idx FEF2575%FEV1', 'idx best FEV1', 'Airway resistance (%)',
       'Airway conductance (%)', 'mean AC', 'P(ppFEV1|AC)',
       'P(ppFEV1|AC) ratioed', 'P(ppFEV1|AC ratioed)',
       'clipped ecFEV1%Predicted', 'Avg Home IVs', 'Avg Hosp IVs', 'Avg IVs',
       'Avg Oral', 'Avg Any antibiotics', 'AC sampled', 'diff (clipped)'],
      dtype='object')

In [ ]:
df_conf["diff (clipped)"] = df_conf["mean AC"] - df_conf["clipped ecFEV1%Predicted"]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_19058/134932417.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
idx_large_diff = df_conf["P(ppFEV1|AC)"].sort_values(ascending=True).head(20).index
cols2keep = [
    "ID",
    "Sex",
    "Age",
    "Height",
    "FEV1",
    "Predicted FEV1",
    "FEF2575",
    "ecFEF2575%ecFEV1",
    "clipped ecFEV1%Predicted",
    "mean AC",
    "diff (clipped)",
    "P(ppFEV1|AC)",
    "Avg Any antibiotics",
]
df_conf[cols2keep].loc[idx_large_diff].sort_values(by="diff (clipped)", ascending=False)

,ID,Sex,Age,Height,FEV1,Predicted FEV1,FEF2575,ecFEF2575%ecFEV1,clipped ecFEV1%Predicted,mean AC,diff (clipped),P(ppFEV1|AC),Avg Any antibiotics
506,B158714,Female,38,185,2.24,4.031068,0.84,37.499999,55.568404,92.260493,36.692089,1.573554e-14,0.333333
2032,C222546,Female,65,182,1.45,2.992230,0.53,36.551721,48.458848,81.442276,32.983428,1.212774e-07,1.000000
776,B161137,Female,21,169,2.01,3.611874,0.95,47.263681,55.649777,87.670504,32.020726,1.126647e-10,0.600000
1016,B162277,Female,18,183,3.86,4.269905,4.41,114.248704,90.400141,77.712726,-12.687415,1.864928e-10,0.800000
574,B159340,Female,22,176,4.11,3.928017,3.33,81.021893,100.000000,86.646572,-13.353428,6.477634e-07,1.000000
1342,B163943,Female,37,179,3.75,3.786164,3.27,87.199999,99.044839,85.126222,-13.918618,2.134175e-07,2.600000
1831,B169603,Female,30,179,3.35,3.941105,2.57,76.716418,85.001536,71.004430,-13.997106,7.561220e-12,0.500000
708,B160496,Female,19,175,3.69,3.892887,1.58,42.818429,94.788261,80.516846,-14.271415,1.076870e-07,1.000000
288,B157654,Female,41,183,4.15,3.847642,3.42,82.409638,100.000000,85.573716,-14.426284,2.417545e-09,0.600000
2024,C222164,Female,29,166,3.27,3.373554,3.25,99.388380,96.930422,82.371299,-14.559123,1.127210e-08,1.000000


In [ ]:
idx_large_diff = (
    df_conf["diff (clipped)"].abs().sort_values(ascending=False).head(20).index
)
cols2keep = [
    "ID",
    "Sex",
    "Age",
    "Height",
    "FEV1",
    "Predicted FEV1",
    "FEF2575",
    "ecFEF2575%ecFEV1",
    "clipped ecFEV1%Predicted",
    "mean AC",
    "diff (clipped)",
    "P(ppFEV1|AC)",
]
df_conf[cols2keep].loc[idx_large_diff].sort_values(by="diff (clipped)", ascending=False)

,ID,Sex,Age,Height,FEV1,Predicted FEV1,FEF2575,ecFEF2575%ecFEV1,clipped ecFEV1%Predicted,mean AC,diff (clipped),P(ppFEV1|AC)
506,B158714,Female,38,185,2.24,4.031068,0.84,37.499999,55.568404,92.260493,36.692089,1.573554e-14
72,B156254,Female,45,181,1.93,3.632932,4.33,224.352334,53.125137,86.944227,33.819090,3.205089e-06
2032,C222546,Female,65,182,1.45,2.992230,0.53,36.551721,48.458848,81.442276,32.983428,1.212774e-07
776,B161137,Female,21,169,2.01,3.611874,0.95,47.263681,55.649777,87.670504,32.020726,1.126647e-10
219,B157369,Female,26,160,0.89,3.159282,0.40,44.943822,28.170961,56.324774,28.153812,1.041949e-06
508,B158720,Female,30,153,1.57,2.825143,2.56,163.057316,55.572408,79.488189,23.915781,2.249587e-03
1862,B170362,Female,48,172,2.12,3.175287,0.87,41.037738,66.765607,90.127301,23.361693,1.423226e-06
2036,C222614,Female,40,145,1.76,2.367586,1.14,64.772727,74.337311,93.228145,18.890834,2.003711e-03
385,B158151,Female,18,156,1.64,3.043475,2.42,147.560982,53.885768,72.023652,18.137883,6.579071e-03
1800,B169105,Female,37,164,1.37,3.144692,1.71,124.817521,43.565473,61.134284,17.568811,4.848477e-04


In [101]:
df.columns

Index(['ID', 'Date Recorded', 'Airway resistance (%)', 'ecFEV1 % Predicted',
       'Avg Home IVs', 'Avg Hosp IVs', 'Avg IVs', 'Avg Oral',
       'Avg Any antibiotics', 'Airway conductance (%)', 'mean AC',
       'P(ppFEV1|AC)', 'P(ppFEV1|AC ratioed)'],
      dtype='object')

## Computing mean AC vs ppFEV1 diffs

In [ ]:
# diff = Predicted AC - baseline FEV1%pred
df["diff"] = df["mean AC"] - df["ecFEV1 % Predicted"]

df["clipped ecFEV1%Predicted"] = df["ecFEV1 % Predicted"].clip(upper=100)
df["diff (clipped)"] = df["mean AC"] - df["clipped ecFEV1%Predicted"]

In [37]:
df["AC std"] = df[AC.name].apply(lambda ac: AC.get_std(ac))
df["Large diff"] = abs(df["diff (clipped)"]) > df["AC std"]
df["Below 100%"] = df["ecFEV1 % Predicted"] < 100

In [38]:
import plotly.express as px

# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

# for col in ["Hosp IVs", "Home IVs", "Any antibiotics", "Oral"]:
for col in ["Any antibiotics"]:
    title = f"Association of model output diff against baseline with {col} (2019-23)"
    fig = px.scatter(
        df_plot,
        x=f"diff{xcol}",
        y=f"Avg {col}",
        labels={
            f"diff{xcol}": f"Predicted conductance - Baseline ppFEV1{xcol}",
            f"Avg {col}": f"Average number of {col}",
        },
        # marginal_y="histogram",
        title=title,
        size_max=6,  # controls the maximum bubble size
        size=[3] * len(df_plot),
        hover_data=["ID"],  # Add this line to include df.ID in hover label
    )
    # fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=600, width=800)
    fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
# Why so many individuals with worse lungs have no hosp IVs?

# Home and hops IVs have a similar pattern

# Inaccuracies
# 1. Deal with individuals that have FEV1% > 100%

# Biases
# More people with negative diff
# 2. Few individuals have 3+ IVs

# TODO
# 1. Show marked difference by excluding IDs when baseline is within 1 sigma from prediction
# 2. Compute percentage of people per number of IVs?
# Verify the pattern on other years
# Plot home/hosp IVs between 2019 and 2023. Max 20 per year. Check IV data completeness over the years, if missing values then compute an average number

In [ ]:
# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

# Create the three dataframes
df_mild = df_plot[df_plot["mean AC"] >= 70]
df_moderate = df_plot[(df_plot["mean AC"] >= 40) & (df_plot["mean AC"] < 70)]
df_severe = df_plot[(df_plot["mean AC"] < 40)]

for col in ["Hosp"]:  # , "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"
    fig = make_subplots(rows=1, cols=3)

    for i, dftmp in enumerate([df_mild, df_moderate, df_severe]):
        fig.add_trace(
            go.Scatter(
                x=dftmp[f"diff{xcol}"],
                y=dftmp[f"Avg {col} IVs"],
                mode="markers",
                marker=dict(color="#0072b2", size=3),
            ),
            row=1,
            col=i + 1,
        )

    # fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=500, width=1000)
    # fig.show()
    fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/Plot 1.pdf")

In [54]:
from scipy.stats import pearsonr

# Compute correlation between 'ecFEV1 % Predicted' and 'Avg Hosp IVs'
x1 = df["ecFEV1 % Predicted"]
y1 = df["Avg Hosp IVs"]
corr1, pval1 = pearsonr(x1, y1)

# Compute correlation between 'mean AC' and 'Avg Hosp IVs'
x2 = df["mean AC"]
y2 = df["Avg Hosp IVs"]
corr2, pval2 = pearsonr(x2, y2)

print(f"corr1, pval1: {corr1:.3f}, {pval1:.3g}")
print(f"corr2, pval2: {corr2:.3f}, {pval2:.3g}")

corr1, pval1: -0.376, 1.53e-69
corr2, pval2: -0.380, 2.22e-71


In [ ]:
import plotly.express as px
import numpy as np

# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

for col in ["Hosp", "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_plot["clipped ecFEV1%Predicted"],
            y=df_plot[f"Avg {col} IVs"],
            mode="markers",
        )
    )
    fig.add_trace(
        go.Scatter(x=df_plot["mean AC"], y=df_plot[f"Avg {col} IVs"], mode="markers")
    )

    fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=800, width=800)
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")